In [33]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from tensorflow.keras import Sequential, Input, layers
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from keras.ops import expand_dims

from sklearn.model_selection import train_test_split


In [22]:
# Splitting train and test sets with both datasets and reshaping both X_train and X_test

train = pd.read_csv('../data/preprocessed/Training/processed_training_data.csv')
test  = pd.read_csv('../data/preprocessed/Testing/processed_training_data.csv')
validation  = pd.read_csv('../data/preprocessed/Validation/processed_training_data.csv')
X_train = train.loc[:, "pixel1":"pixel4096"].values.reshape(1008, 64, 64, 1) / 255.0
y_train = train.loc[:, "label"].values
X_val = validation.loc[:, "pixel1":"pixel4096"].values.reshape(168, 64, 64, 1) / 255.0
y_val = validation.loc[:, "label"].values
X_test = test.loc[:, "pixel1":"pixel4096"].values.reshape(501, 64, 64, 1) / 255.0
y_test = test.loc[:, "label"].values

In [21]:
validation.describe

<bound method NDFrame.describe of     label  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  \
0       X     138     142     146     148     149     148     149     151   
1       O     141     142     146     146     151     153     154     159   
2       C     181     181     184     182     183     184     185     188   
3       M     174     176     178     178     180     181     182     183   
4       Y     169     170     171     171     172     174     176     178   
..    ...     ...     ...     ...     ...     ...     ...     ...     ...   
163     L     135     137     140     143     144     145     146     149   
164     U     142     143     144     144     148     149     151     153   
165     N      86      93     101     110     119     123     132     136   
166     V     112     122     126     130     136     138     142     141   
167     Q     125     130     137     143     149     150     153     155   

     pixel9  ...  pixel4087  pixel4088  p

In [ ]:
# Create a validation split on the test set

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [23]:
letter_to_index = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
    "F": 5,
    "G": 6,
    "H": 7,
    "I": 8,
    "J": 9,
    "K": 10,
    "L": 11,
    "M": 12,
    "N": 13,
    "O": 14,
    "P": 15,
    "Q": 16,
    "R": 17,
    "S": 18,
    "T": 19,
    "U": 20,
    "V": 21,
    "W": 22,
    "X": 23,
    "Y": 24,
    "Z": 25
}

In [24]:
y_train = np.array([letter_to_index[char] for char in y_train])


In [25]:
y_val = np.array([letter_to_index[char] for char in y_val])


In [26]:
y_test = np.array([letter_to_index[char] for char in y_test])


## Model creation, fine-tuning, training

In [ ]:
def initialize_model(X_train):

    model = Sequential()

    model.add(Input(shape=X_train.shape[1:]))

    ### First Convolution & MaxPooling
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Second Convolution & MaxPooling
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Third Convolution & MaxPooling
    model.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool2D(pool_size=(2, 2)))

    ### Flattening
    model.add(layers.Flatten())

    ### One Fully Connected layer - "Fully Connected" is equivalent to saying "Dense"
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.BatchNormalization())

    ### Dropout layer - rate = 0.3, fraction of units to drop
    model.add(layers.Dropout(0.3))

    ### Last layer - Classification Layer with 25 outputs corresponding to 25 digits
    model.add(layers.Dense(25, activation='softmax'))

    ### Model compilation
    model.compile(
        loss='sparse_categorical_crossentropy', # sparse if not to_categorical
        optimizer='adam',
        metrics=['accuracy'])

    return model


In [36]:
# Calling model
model = initialize_model(X_train)
adam_opt = optimizers.Adam(learning_rate=0.001)

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=adam_opt,
              metrics=['accuracy'])


In [29]:
# Model Summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 64, 64, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 30, 30, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 30, 30, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 15, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 13, 13, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 13, 13, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 25)             │         1,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 132,345 (516.97 KB)

 Trainable params: 131,897 (515.22 KB)

 Non-trainable params: 448 (1.75 KB)

In [39]:
# Call EarlyStopping and train model
lr = ReduceLROnPlateau(monitor = 'val_loss', factor=0.1, patience=5, verbose=1, min_lr=0)

es = EarlyStopping(patience=5, monitor='val_loss', verbose=1, restore_best_weights=True)

history = model.fit(X_train, y_train,
         batch_size=64,
         epochs=200,
         callbacks=[lr],
         validation_data=(X_val, y_val),
         verbose=1)

Epoch 1/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.5496 - loss: 1.6117 - val_accuracy: 0.0476 - val_loss: 3.2503 - learning_rate: 0.0010
Epoch 2/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.8006 - loss: 0.9780 - val_accuracy: 0.0476 - val_loss: 3.2803 - learning_rate: 0.0010
Epoch 3/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - accuracy: 0.9008 - loss: 0.6056 - val_accuracy: 0.0833 - val_loss: 3.3048 - learning_rate: 0.0010
Epoch 4/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.9474 - loss: 0.4037 - val_accuracy: 0.0536 - val_loss: 3.3452 - learning_rate: 0.0010
Epoch 5/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.9802 - loss: 0.2743 - val_accuracy: 0.1012 - val_loss: 3.3800 - learning_rate: 0.0010
Epoch 6/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9782 - loss: 0.2208
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
16/16 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.9782 - loss: 0.2208 - val_accur

KeyboardInterrupt: 

In [40]:
model.evaluate(X_test, y_test, verbose=1)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8483 - loss: 0.7175


[0.7174543142318726, 0.8483033776283264]

In [41]:
path = "/home/maxgr/code/notFonz/HandSpell/data/raw_data/A02_original_aslmnist/Dataset/user_3/A3.jpg"


In [43]:
model.save('model_64.keras')

Things to add:
- Embedding layer

Things to ask:
- Do we need masking here ? aka embedding layer

In [9]:
# Import model as pkl file on models
import joblib

with open("mnist_baseline_model", "wb") as file:
    joblib.dump(model, file)